# LLM-Assisted Fraud Explanation Layer

This notebook is **downstream of `02_ml_risk_scoring.ipynb`**.

Notebook 02 already decides:
- the selected model,
- the validation-selected review threshold,
- and the final untouched-test evaluation.

This notebook does **not** compare models or tune thresholds again. Its only job is to explain high-risk records that the final risk-scoring policy sends to manual review.

Workflow:

`02 outputs → fixed Random Forest + fixed threshold → Review records → SHAP local signals → LLM analyst notes → faithfulness checks`


## 1. Setup


In [16]:
# Run once if needed:
# pip install shap openai

from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import shap

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from openai import OpenAI

DATA_PATH = Path("../data/raw/creditcard.csv")
PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../reports/llm_explanations")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load the final decisions from Notebook 02

We reuse the model and threshold choices already made in Notebook 02.  
There is no model comparison or threshold search here.


In [17]:
final_test_metrics = pd.read_csv(PROCESSED_DIR / "final_test_metrics.csv")

if len(final_test_metrics) != 1:
    raise ValueError("Expected exactly one row in final_test_metrics.csv")

final_row = final_test_metrics.iloc[0]

selected_model = final_row["model"]
selected_threshold = float(final_row["selected_threshold_from_validation"])

print("Selected model from Notebook 02:", selected_model)
print("Validation-selected threshold from Notebook 02:", selected_threshold)

display(final_test_metrics)


Selected model from Notebook 02: Random Forest
Validation-selected threshold from Notebook 02: 0.5


,split,model,selected_threshold_from_validation,precision,recall,f1,average_precision,pr_auc_trapezoid,accuracy_reference,roc_auc_reference
0,test,Random Forest,0.5,0.82,0.836735,0.828283,0.793414,0.794728,0.999403,0.977921


## 3. Recreate the selected Random Forest for explanation

Notebook 02 currently saves the final metrics and scored sample, but not a serialized model object.  
To compute SHAP values, this notebook recreates the **same deterministic Random Forest** using the same training split and fixed hyperparameters from Notebook 02.

Important: this is **not model selection**. We do not train Logistic Regression, compare AP, inspect validation performance, or retune the threshold. The model and threshold are already fixed by Notebook 02.


In [18]:
import joblib

MODEL_PATH = Path("../models/random_forest_final.joblib")

if selected_model != "Random Forest":
    raise ValueError(
        f"This explanation notebook currently expects Random Forest, "
        f"but Notebook 02 selected {selected_model!r}."
    )

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=["Class"])
y = df["Class"]

# Reproduce the same test split used in Notebook 02.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Load the already-trained final model from Notebook 02.
rf_model = joblib.load(MODEL_PATH)

test_scores = rf_model.predict_proba(X_test)[:, 1]

test_decisions = np.where(
    test_scores >= selected_threshold,
    "Review",
    "Pass"
)

test_output = X_test.copy()
test_output["record_id"] = test_output.index
test_output["actual_class"] = y_test.values
test_output["fraud_score"] = test_scores
test_output["selected_threshold"] = selected_threshold
test_output["decision"] = test_decisions

## 4. Select high-risk Review records

Only records already sent to **Review** by the fixed policy are candidates for explanation.  
The 50 highest fraud scores are used for SHAP analysis.


In [19]:
flagged = (
    test_output[test_output["decision"] == "Review"]
    .sort_values("fraud_score", ascending=False)
    .head(50)
    .reset_index(drop=True)
)

display(
    flagged[
        ["record_id", "Amount", "fraud_score",
         "selected_threshold", "decision", "actual_class"]
    ].head(10)
)

flagged.to_csv(
    OUTPUT_DIR / "flagged_transactions.csv",
    index=False
)


,record_id,Amount,fraud_score,selected_threshold,decision,actual_class
0,42009,112.33,0.999531,0.5,Review,1
1,44556,1.00,0.999520,0.5,Review,1
2,119781,124.53,0.999518,0.5,Review,1
3,143728,1.00,0.999517,0.5,Review,1
4,150665,209.65,0.999514,0.5,Review,1
5,150647,8.54,0.999512,0.5,Review,1
6,42769,118.30,0.999510,0.5,Review,1
7,150663,316.06,0.999510,0.5,Review,1
8,42756,0.00,0.999507,0.5,Review,1
9,42696,88.23,0.999507,0.5,Review,1


## 5. Build local risk signals with SHAP

Global feature importance answers “which features matter overall.”

Here we need a **local explanation** for each specific reviewed transaction.  
SHAP identifies which anonymized model features pushed that record's fraud score upward.

Because `V1`–`V28` are anonymized PCA features, we do not assign them real-world meanings.


In [20]:
model_columns = list(rf_model.feature_names_in_)

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(flagged[model_columns])

# Handle common SHAP output formats for binary classifiers.
if isinstance(shap_values, list):
    fraud_shap_values = shap_values[1]
elif hasattr(shap_values, "ndim") and shap_values.ndim == 3:
    fraud_shap_values = shap_values[:, :, 1]
else:
    fraud_shap_values = shap_values

signal_features = []
local_signals = []

for i in range(len(flagged)):
    contributions = pd.Series(
        fraud_shap_values[i],
        index=model_columns
    )

    # Keep the three strongest positive contributions:
    # features pushing the fraud prediction upward.
    top_positive = (
        contributions[contributions > 0]
        .sort_values(ascending=False)
        .head(3)
    )

    feature_names = list(top_positive.index)
    signal_features.append(", ".join(feature_names))

    signal_text = []
    for feature in feature_names:
        feature_value = flagged.loc[i, feature]
        shap_value = top_positive[feature]
        signal_text.append(
            f"{feature}={feature_value:.3f} "
            f"(SHAP contribution +{shap_value:.3f})"
        )

    local_signals.append("; ".join(signal_text))

flagged["signal_features"] = signal_features
flagged["local_signals"] = local_signals

explanation_input = flagged[
    [
        "record_id",
        "Amount",
        "fraud_score",
        "selected_threshold",
        "decision",
        "signal_features",
        "local_signals"
    ]
].copy()

display(explanation_input.head())

explanation_input.to_csv(
    OUTPUT_DIR / "flagged_transaction_explanations_input.csv",
    index=False
)


,record_id,Amount,fraud_score,selected_threshold,decision,signal_features,local_signals
0,42009,112.33,0.999531,0.5,Review,"V14, V12, V17",V14=-8.226 (SHAP contribution +0.126); V12=-7....
1,44556,1.00,0.999520,0.5,Review,"V14, V10, V17",V14=-7.246 (SHAP contribution +0.129); V10=-5....
2,119781,124.53,0.999518,0.5,Review,"V14, V10, V12",V14=-7.495 (SHAP contribution +0.133); V10=-5....
3,143728,1.00,0.999517,0.5,Review,"V14, V10, V12",V14=-8.604 (SHAP contribution +0.130); V10=-6....
4,150665,209.65,0.999514,0.5,Review,"V14, V10, V12",V14=-11.030 (SHAP contribution +0.130); V10=-1...


## 6. Generate analyst-facing explanations with an LLM

The Random Forest makes the risk decision.  
The LLM does **not** decide whether a transaction is fraud.

It only turns verified inputs — score, threshold, amount, and SHAP signals — into a short analyst note.


In [21]:
OPENAI_API_KEY =  " Enter your API key:"
client = OpenAI(api_key=OPENAI_API_KEY)

def generate_explanation(row):
    prompt = f"""
Write a short note for a fraud analyst using only the facts below.

Rules:
- State that the record was sent to manual review.
- Give the fraud score and review threshold exactly as shown.
- Mention the three local model features exactly as shown.
- Do not assign real-world meanings to anonymized features.
- Do not mention or infer merchant, cardholder, device, IP address,
  identity, location, country, or city.
- Do not claim that the transaction is definitely fraud.
- Write 2 or 3 simple sentences.

Record ID: {row['record_id']}
Fraud score: {row['fraud_score']:.3f}
Review threshold: {row['selected_threshold']:.2f}
Transaction amount: {row['Amount']:.2f}
Local model signals: {row['local_signals']}
"""

    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt,
        store=False
    )
    return response.output_text.strip()

# Generate a small demonstration set rather than sending all records to the API.
explanations = explanation_input.head(10).copy()
explanations["analyst_explanation"] = explanations.apply(
    generate_explanation,
    axis=1
)

display(
    explanations[
        ["record_id", "fraud_score", "decision", "analyst_explanation"]
    ]
)

explanations.to_csv(
    OUTPUT_DIR / "analyst_explanations.csv",
    index=False
)


,record_id,fraud_score,decision,analyst_explanation
0,42009,0.999531,Review,Record ID: 42009 was sent to manual review. Fr...
1,44556,0.999520,Review,Record 44556 was sent to manual review. \nFra...
2,119781,0.999518,Review,Record ID 119781 was sent to manual review. Fr...
3,143728,0.999517,Review,Record 143728 was sent to manual review. Fraud...
4,150665,0.999514,Review,Record 150665 was sent to manual review. Fraud...
5,150647,0.999512,Review,Record 150647 was sent to manual review. Fraud...
6,42769,0.999510,Review,Record 42769 was sent to manual review. Fraud ...
7,150663,0.999510,Review,Record ID: 150663 was sent to manual review. F...
8,42756,0.999507,Review,Record 42756 was sent to manual review. Fraud ...
9,42696,0.999507,Review,Record 42696 was sent to manual review. \nFra...


## 7. Basic faithfulness checks

These checks do not prove an explanation is perfect.  
They catch simple failures such as missing the score/threshold, omitting the SHAP feature names, or introducing unsupported real-world entities.


In [22]:
unsupported_terms = [
    "merchant",
    "cardholder",
    "device",
    "ip address",
    "identity",
    "location",
    "country",
    "city"
]

check_rows = []

for _, row in explanations.iterrows():
    text = row["analyst_explanation"].lower()

    score_is_present = f"{row['fraud_score']:.3f}" in text
    threshold_is_present = f"{row['selected_threshold']:.2f}" in text

    feature_names = [
        feature.strip()
        for feature in row["signal_features"].split(",")
        if feature.strip()
    ]
    features_are_present = all(
        feature.lower() in text
        for feature in feature_names
    )

    found_unsupported_term = any(
        term in text
        for term in unsupported_terms
    )

    passed = (
        score_is_present
        and threshold_is_present
        and features_are_present
        and not found_unsupported_term
    )

    check_rows.append({
        "record_id": row["record_id"],
        "score_present": score_is_present,
        "threshold_present": threshold_is_present,
        "local_features_present": features_are_present,
        "unsupported_claim_flag": found_unsupported_term,
        "passed_basic_checks": passed,
        "analyst_explanation": row["analyst_explanation"]
    })

eval_table = pd.DataFrame(check_rows)

display(eval_table)

eval_table.to_csv(
    OUTPUT_DIR / "llm_explanation_faithfulness_review.csv",
    index=False
)

print(
    "Passed basic checks:",
    int(eval_table["passed_basic_checks"].sum()),
    "out of",
    len(eval_table)
)


,record_id,score_present,threshold_present,local_features_present,unsupported_claim_flag,passed_basic_checks,analyst_explanation
0,42009,True,True,True,False,True,Record ID: 42009 was sent to manual review. Fr...
1,44556,True,True,True,False,True,Record 44556 was sent to manual review. \nFra...
2,119781,True,True,True,False,True,Record ID 119781 was sent to manual review. Fr...
3,143728,True,True,True,False,True,Record 143728 was sent to manual review. Fraud...
4,150665,True,True,True,False,True,Record 150665 was sent to manual review. Fraud...
5,150647,True,True,True,False,True,Record 150647 was sent to manual review. Fraud...
6,42769,True,True,True,False,True,Record 42769 was sent to manual review. Fraud ...
7,150663,True,True,True,False,True,Record ID: 150663 was sent to manual review. F...
8,42756,True,True,True,False,True,Record 42756 was sent to manual review. Fraud ...
9,42696,True,True,True,False,True,Record 42696 was sent to manual review. \nFra...


Passed basic checks: 10 out of 10


## 8. Project summary


In [23]:
summary_text = f"""
Explanation-layer summary:

Notebook 02 already selected the final fraud model and review threshold.
This notebook reused those fixed choices instead of repeating model comparison
or threshold tuning.

Selected model: {selected_model}
Validation-selected threshold: {selected_threshold:.2f}
Test records sent to Review: {(test_output["decision"] == "Review").sum()}
High-risk Review records prepared for SHAP: {len(flagged)}
LLM explanations generated: {len(explanations)}
Explanations passing basic checks: {int(eval_table["passed_basic_checks"].sum())} out of {len(eval_table)}

Explanation workflow:
fixed risk policy -> Review records -> local SHAP signals
-> constrained LLM analyst notes -> basic faithfulness checks
"""

print(summary_text)

(OUTPUT_DIR / "project_summary.txt").write_text(summary_text)



Explanation-layer summary:

Notebook 02 already selected the final fraud model and review threshold.
This notebook reused those fixed choices instead of repeating model comparison
or threshold tuning.

Selected model: Random Forest
Validation-selected threshold: 0.50
Test records sent to Review: 100
High-risk Review records prepared for SHAP: 50
LLM explanations generated: 10
Explanations passing basic checks: 10 out of 10

Explanation workflow:
fixed risk policy -> Review records -> local SHAP signals
-> constrained LLM analyst notes -> basic faithfulness checks



571